# Abandoned boat detection – Istanbul pilot

Detects boats in every image, lines up the years of each site, tracks each boat over time and
flags **abandonment candidates**: boats in (approximately) the same position for **≥ 5 years**,
following Payne & Turner (2023).

**Before you start**
1. Upload the folder `abandoned ships` to your Google Drive (e.g. `My Drive/abandoned ships`).
2. Menu **Runtime → Change runtime type → T4 GPU → Save**.
3. Check the two folder paths in step 2, then **Runtime → Run all**.

Results are written to your Drive, and finished images are cached — if Colab disconnects,
just run all cells again and it continues where it stopped.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - go to Runtime > Change runtime type > T4 GPU"

## 2. Connect Google Drive and set folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/abandoned ships'          # folder with the 237 images
OUT_DIR  = '/content/drive/MyDrive/abandoned ships results'  # results are written here

import os
n = len([f for f in os.listdir(DATA_DIR) if f.lower().endswith('.jpg')])
print(f'{n} images found in {DATA_DIR}')

## 3. Install the detector

In [ ]:
!pip -q install ultralytics

## 4. Write the pipeline code
(You don't need to change anything here.)

In [ ]:
%%writefile boat_pipeline.py
#!/usr/bin/env python3
"""
boat_pipeline.py - detect and track boats across multi-date Google Earth exports
and flag abandonment candidates, following Payne & Turner (2023):
  "Boats were defined as being abandoned if they had been in precisely (or
   approximately) the same position for a period of at least 5 years ..."

Input : a folder of images named  <SiteNo>_<Site>_<YYYY-MM>.jpg
Output: OUT_DIR/
          alignment.csv          shift of every image relative to its site's reference image
          detections.csv         every detected boat in every image
          tracks.csv / .xlsx     one row per tracked boat, with status + empty review columns
          review/<site>/         image strips of each candidate across all years
          review/<site>_overview.jpg   map of candidates on the reference image
          cache/                 per-image detections (re-runs skip finished images)

Stages are cached, so the script can be stopped and resumed.
"""

import argparse
import json
import math
import re
import sys
import time
from collections import defaultdict
from datetime import date
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ settings
DEFAULTS = dict(
    model="yolo11m-obb.pt",   # DOTA-trained oriented-box detector, class 1 = ship
    conf=0.25,                # detection confidence threshold
    tile=1024, overlap=256,   # full-resolution tiling
    batch=8,
    default_m_per_px=0.143,   # fallback if the scale bar can't be read (50 m bar ~ 350 px)
    scale_bar_m=50.0,         # length printed on Google Earth's scale bar
    match_m=8.0,              # max movement (m) to count as "same position" between dates
    min_years=5.0,            # abandonment rule from the paper
    min_presence=0.6,         # share of images between first/last sighting where it's detected
    min_len_m=2.5,            # ignore detections shorter than this (paper min = 2.6 m)
    appearance_thr=0.55,      # below this, a "persistent" spot probably holds different boats
    max_spread_m=4.0,         # candidate: position varies less than this across years
    max_len_change=0.25,      # candidate: length varies less than 25 %
    max_angle_change=20.0,    # candidate: heading varies less than 20 degrees
    absent_ratio=0.45,        # spot counts as empty water if its texture < 45 % of the boat's
)

# review priority: 1 = most likely abandoned, 2 = could be a decaying boat (appearance changes)
PRIORITY = {"CANDIDATE abandoned": 1, "same spot, appearance changed - check": 2}

FNAME_RE = re.compile(r"^(?P<siteno>[\d\-]+)_(?P<site>.+)_(?P<y>\d{4})-(?P<m>\d{2})$")


# ------------------------------------------------------------------ helpers
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def index_images(folder):
    rows = []
    for p in sorted(Path(folder).iterdir()):
        if p.suffix.lower() not in (".jpg", ".jpeg", ".png", ".tif", ".tiff"):
            continue
        m = FNAME_RE.match(p.stem)
        if not m:
            log(f"  skipped (name not <No>_<Site>_<YYYY-MM>): {p.name}")
            continue
        y, mo = int(m["y"]), int(m["m"])
        rows.append(dict(file=p.name, path=str(p), site_id=f'{m["siteno"]}_{m["site"]}',
                         site_no=m["siteno"], site=m["site"], date=f"{y}-{mo:02d}",
                         t=y + (mo - 0.5) / 12))
    if not rows:
        sys.exit(f"No images found in {folder}")
    return pd.DataFrame(rows).sort_values(["site_id", "t"]).reset_index(drop=True)


def overlay_mask_boxes(w, h):
    """Google Earth overlays: logo/credits bottom-left, scale bar + north arrow bottom-right."""
    return [(0, int(h * 0.955), int(w * 0.06), h),
            (int(w * 0.93), int(h * 0.94), w, h)]


def in_boxes(x, y, boxes):
    return any(x0 <= x <= x1 and y0 <= y <= y1 for x0, y0, x1, y1 in boxes)


def read_scale_bar(img, bar_m):
    """Length of the white scale bar (bottom-right) -> metres per pixel, or None."""
    h, w = img.shape[:2]
    reg = img[h - 70:h, w - 900:w]
    white = np.all(reg > 215, axis=2)
    best = 0
    for row in white:
        # longest run of white pixels in this row
        r = np.diff(np.concatenate(([0], row.view(np.int8), [0])))
        starts, ends = np.where(r == 1)[0], np.where(r == -1)[0]
        if len(starts):
            best = max(best, int((ends - starts).max()))
    return bar_m / best if 150 < best < 880 else None


# ------------------------------------------------------------------ 1. alignment
def edge_image(path, reduce):
    flag = {2: cv2.IMREAD_REDUCED_GRAYSCALE_2, 4: cv2.IMREAD_REDUCED_GRAYSCALE_4}[reduce]
    g = cv2.imread(path, flag).astype(np.float32)
    g = cv2.GaussianBlur(g, (0, 0), 1.5)
    e = np.log1p(np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)))
    h, w = e.shape
    for x0, y0, x1, y1 in overlay_mask_boxes(w, h):
        e[y0:y1, x0:x1] = 0
    return e - e.mean()


def phase_shift(ref, mov):
    win = cv2.createHanningWindow(ref.shape[::-1], cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(mov, ref, win)
    return dx, dy, resp


def align_site(rows, ref_path):
    """Translation of every image onto the reference (Google Earth views here are
    north-up at a fixed zoom, so a shift is all that differs)."""
    out = {}
    ref4, ref2 = edge_image(ref_path, 4), edge_image(ref_path, 2)
    for _, r in rows.iterrows():
        if r.path == ref_path:
            out[r.file] = (0.0, 0.0, 1.0)
            continue
        dx, dy, resp = phase_shift(ref4, edge_image(r.path, 4))
        dx, dy = dx * 4, dy * 4
        # refine at half resolution
        mov2 = edge_image(r.path, 2)
        M = np.float32([[1, 0, dx / 2], [0, 1, dy / 2]])
        mov2 = cv2.warpAffine(mov2, M, mov2.shape[::-1])
        ddx, ddy, resp2 = phase_shift(ref2, mov2)
        if abs(ddx) < 20 and abs(ddy) < 20:
            dx, dy, resp = dx + ddx * 2, dy + ddy * 2, resp2
        out[r.file] = (float(dx), float(dy), float(resp))
    return out


# ------------------------------------------------------------------ 2. detection
class Detector:
    def __init__(self, cfg):
        from ultralytics import YOLO
        self.m = YOLO(cfg["model"])
        self.cfg = cfg
        self.ship = [k for k, v in self.m.names.items() if v == "ship"]

    def detect(self, img):
        c = self.cfg
        h, w = img.shape[:2]
        T, step = c["tile"], c["tile"] - c["overlap"]
        xs = list(range(0, max(w - T, 0) + 1, step)) or [0]
        ys = list(range(0, max(h - T, 0) + 1, step)) or [0]
        if xs[-1] + T < w: xs.append(w - T)
        if ys[-1] + T < h: ys.append(h - T)
        tiles = [(x, y) for y in ys for x in xs]
        boxes, scores = [], []
        for i in range(0, len(tiles), c["batch"]):
            chunk = tiles[i:i + c["batch"]]
            res = self.m.predict([img[y:y + T, x:x + T] for x, y in chunk], imgsz=T,
                                 conf=c["conf"], classes=self.ship, verbose=False)
            for (x0, y0), r in zip(chunk, res):
                if r.obb is None or len(r.obb) == 0:
                    continue
                xywhr = r.obb.xywhr.cpu().numpy()
                corners = r.obb.xyxyxyxy.cpu().numpy()
                confs = r.obb.conf.cpu().numpy()
                for (cx, cy, bw, bh, rot), cor, cf in zip(xywhr, corners, confs):
                    # drop boxes cut by an internal tile border (the overlap sees them whole)
                    cut = False
                    if x0 > 0 and cor[:, 0].min() < 3: cut = True
                    if y0 > 0 and cor[:, 1].min() < 3: cut = True
                    if x0 + T < w and cor[:, 0].max() > T - 3: cut = True
                    if y0 + T < h and cor[:, 1].max() > T - 3: cut = True
                    if cut:
                        continue
                    boxes.append(((float(cx + x0), float(cy + y0)), (float(bw), float(bh)),
                                  float(math.degrees(rot))))
                    scores.append(float(cf))
        if not boxes:
            return []
        keep = cv2.dnn.NMSBoxesRotated(boxes, scores, c["conf"], 0.3)
        keep = np.array(keep).flatten()
        mask = overlay_mask_boxes(w, h)
        dets = []
        for k in keep:
            (cx, cy), (bw, bh), ang = boxes[k]
            if in_boxes(cx, cy, mask):
                continue
            dets.append(dict(cx=cx, cy=cy, len_px=max(bw, bh), wid_px=min(bw, bh),
                             angle=ang if bw >= bh else ang + 90, conf=scores[k]))
        return dets


# ------------------------------------------------------------------ 3. appearance
class Appearance:
    """Similarity of two boat crops. DINOv2 (via torch.hub) when available,
    otherwise a simple orientation-tolerant grey-level correlation."""

    def __init__(self, use_deep=True):
        self.model = None
        if use_deep:
            try:
                import torch
                self.torch = torch
                self.dev = "cuda" if torch.cuda.is_available() else "cpu"
                self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14",
                                            verbose=False).to(self.dev).eval()
                log("  appearance model: DINOv2")
            except Exception as e:
                log(f"  appearance model unavailable ({e.__class__.__name__}); using simple fallback")
        self.method = "dinov2" if self.model is not None else "ncc"

    def embed(self, crop):
        if self.model is None:
            g = cv2.cvtColor(cv2.resize(crop, (64, 64)), cv2.COLOR_BGR2GRAY).astype(np.float32)
            return (g - g.mean()) / (g.std() + 1e-6)
        t = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), (224, 224)).astype(np.float32) / 255
        t = (t - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
        x = self.torch.from_numpy(t.transpose(2, 0, 1)).float()[None].to(self.dev)
        with self.torch.no_grad():
            v = self.model(x)[0]
        return (v / v.norm()).cpu().numpy()

    def sim(self, a, b):
        if self.model is None:
            return float(max((a * np.rot90(b, k)).mean() for k in (0, 2)))  # allow 180° turn
        return float(np.dot(a, b))


def boat_crop(img, cx, cy, ang, length, pad=1.4, out=160):
    """Crop rotated so the boat lies horizontally (comparable across years)."""
    side = max(length * pad, 24)
    M = cv2.getRotationMatrix2D((cx, cy), ang, 1.0)
    M[:, 2] += [side / 2 - cx, side / 2 - cy]
    c = cv2.warpAffine(img, M, (int(side), int(side)), borderValue=(0, 0, 0))
    return cv2.resize(c, (out, out))


def footprint_std(img, cx, cy, ang, length, width):
    """Grey-level texture inside the boat's footprint: high for a boat, low for open water."""
    c = boat_crop(img, cx, cy, ang, length, pad=1.0, out=96)
    g = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY).astype(np.float32)
    h = max(int(96 * width / max(length, 1) * 0.8), 6)
    band = g[48 - h // 2:48 + h // 2, 10:86]  # central strip along the hull
    return float(band.std() + 0.5 * abs(band.mean() - np.median(g)))


# ------------------------------------------------------------------ 4. tracking
def track_site(dets, img_rows, cfg, mpp):
    """Link detections across dates by position in the reference frame."""
    from scipy.optimize import linear_sum_assignment
    gate = cfg["match_m"] / mpp
    tracks = []  # each: dict(id, members=[det idx], x, y, len)
    for _, r in img_rows.iterrows():
        di = dets.index[dets.file == r.file].tolist()
        if not di:
            continue
        if tracks:
            T = np.array([[t["x"], t["y"], t["len"]] for t in tracks])
            D = dets.loc[di, ["rx", "ry", "len_px"]].to_numpy()
            dist = np.hypot(T[:, None, 0] - D[None, :, 0], T[:, None, 1] - D[None, :, 1])
            ratio = np.maximum(T[:, None, 2], D[None, :, 2]) / np.maximum(1e-6, np.minimum(T[:, None, 2], D[None, :, 2]))
            cost = np.where((dist <= gate) & (ratio <= 2.0), dist, 1e9)
            ti, dj = linear_sum_assignment(cost)
            used = set()
            for a, b in zip(ti, dj):
                if cost[a, b] < 1e9:
                    t, d = tracks[a], di[b]
                    t["members"].append(d)
                    t["x"], t["y"] = dets.at[d, "rx"], dets.at[d, "ry"]
                    t["len"] = 0.5 * t["len"] + 0.5 * dets.at[d, "len_px"]
                    used.add(b)
            new = [di[j] for j in range(len(di)) if j not in used]
        else:
            new = di
        for d in new:
            tracks.append(dict(members=[d], x=dets.at[d, "rx"], y=dets.at[d, "ry"],
                               len=dets.at[d, "len_px"]))
    return tracks


# ------------------------------------------------------------------ main
def run(cfg):
    data, out = Path(cfg["data"]), Path(cfg["out"])
    (out / "cache").mkdir(parents=True, exist_ok=True)
    imgs = index_images(data)
    if cfg.get("sites"):
        imgs = imgs[imgs.site_no.isin(cfg["sites"]) | imgs.site_id.isin(cfg["sites"])]
    log(f"{len(imgs)} images, {imgs.site_id.nunique()} sites")

    # ---- alignment + scale
    al_path = out / "alignment.csv"
    al_old = pd.read_csv(al_path).set_index("file") if al_path.exists() else pd.DataFrame()
    al_rows = []
    for site, rows in imgs.groupby("site_id", sort=False):
        ref = rows.iloc[-1]  # most recent image is the reference
        if not al_old.empty and set(rows.file) <= set(al_old.index):
            for f in rows.file:
                al_rows.append(dict(file=f, **al_old.loc[f].to_dict()))
            continue
        log(f"align {site} (ref {ref.date})")
        sh = align_site(rows, ref.path)
        for _, r in rows.iterrows():
            im = cv2.imread(r.path)
            mpp = read_scale_bar(im, cfg["scale_bar_m"])
            dx, dy, q = sh[r.file]
            al_rows.append(dict(file=r.file, site_id=site, ref_file=ref.file, dx=dx, dy=dy,
                                align_quality=q, m_per_px=mpp))
    al = pd.DataFrame(al_rows).set_index("file")
    # keep rows of sites not processed in this run
    keep_old = al_old.loc[[f for f in al_old.index if f not in al.index]] if not al_old.empty else al_old
    pd.concat([keep_old, al]).reset_index().rename(columns={"index": "file"}).to_csv(al_path, index=False)
    imgs = imgs.join(al[["dx", "dy", "align_quality", "m_per_px"]], on="file")
    bad = imgs[imgs.align_quality < 0.025]
    if len(bad):
        log(f"  low alignment confidence for {len(bad)} image(s): " + ", ".join(bad.file))

    # ---- detection (cached per image)
    det = None
    all_dets = []
    for i, r in enumerate(imgs.itertuples(), 1):
        cp = out / "cache" / (Path(r.file).stem + ".json")
        if cp.exists():
            d = json.loads(cp.read_text())
        else:
            if det is None:
                det = Detector(cfg)
            t0 = time.time()
            d = det.detect(cv2.imread(r.path))
            cp.write_text(json.dumps(d))
            log(f"detect [{i}/{len(imgs)}] {r.file}: {len(d)} boats ({time.time() - t0:.0f}s)")
        for x in d:
            x.update(file=r.file, site_id=r.site_id, date=r.date, t=r.t)
        all_dets += d
    dets = pd.DataFrame(all_dets)
    if dets.empty:
        sys.exit("No boats detected.")
    dets = dets.join(imgs.set_index("file")[["dx", "dy", "m_per_px"]], on="file")
    site_mpp = imgs.groupby("site_id").m_per_px.median().fillna(cfg["default_m_per_px"])
    dets["mpp"] = dets.site_id.map(site_mpp)
    dets["len_m"] = dets.len_px * dets.mpp
    dets = dets[dets.len_m >= cfg["min_len_m"]].reset_index(drop=True)
    dets["rx"], dets["ry"] = dets.cx + dets.dx, dets.cy + dets.dy  # reference-frame position

    # ---- tracking + rules
    app = Appearance(use_deep=not cfg.get("no_deep"))
    track_rows = []
    dets["track_id"] = ""
    for site, rows in imgs.groupby("site_id", sort=False):
        sd = dets[dets.site_id == site]
        if sd.empty:
            continue
        mpp = site_mpp[site]
        tracks = track_site(sd, rows, cfg, mpp)
        site_times = rows.t.to_numpy()
        img_cache = {}

        def img(f):
            # keep all images of this site in memory (~1 GB): each is read from Drive only once
            if f not in img_cache:
                img_cache[f] = cv2.imread(str(data / f))
            return img_cache[f]

        cand_n = 0
        strip_jobs = []
        for t in tracks:
            m = sd.loc[t["members"]].sort_values("t")
            t0, t1 = m.t.iloc[0], m.t.iloc[-1]
            n_between = int(((site_times >= t0 - 1e-6) & (site_times <= t1 + 1e-6)).sum())
            span = t1 - t0
            presence = len(m) / n_between
            spread_m = float(np.hypot(m.rx - m.rx.mean(), m.ry - m.ry.mean()).max() * mpp)
            persistent = span >= cfg["min_years"] and presence >= cfg["min_presence"] and len(m) >= 2
            tid = f"{site.split('_')[0]}-{len(track_rows) + 1:04d}"
            dets.loc[m.index, "track_id"] = tid
            appear = np.nan
            len_change = float((m.len_px.max() - m.len_px.min()) / m.len_px.median())
            a = np.radians(m.angle.to_numpy() * 2)  # headings are 180-degree ambiguous
            mean_a = math.atan2(np.sin(a).mean(), np.cos(a).mean())
            angle_change = float(np.degrees(np.abs(np.angle(np.exp(1j * (a - mean_a))))).max() / 2)
            stable = (spread_m <= cfg["max_spread_m"] and len_change <= cfg["max_len_change"]
                      and angle_change <= cfg["max_angle_change"])
            # years inside the period where it was NOT detected: empty water (boat left) or a detector miss?
            absent = []
            if persistent and stable:
                fp = [footprint_std(img(r.file), r.cx, r.cy, r.angle, r.len_px, r.wid_px) for r in m.itertuples()]
                ref_std = float(np.median(fp))
                med_len, med_wid = float(m.len_px.median()), float(m.wid_px.median())
                med_ang = float(m.angle.iloc[-1])
                for r in rows[(rows.t > t0) & (rows.t < t1) & (~rows.file.isin(m.file))].itertuples():
                    s_ = footprint_std(img(r.file), t["x"] - r.dx, t["y"] - r.dy, med_ang, med_len, med_wid)
                    if s_ < cfg["absent_ratio"] * ref_std:
                        absent.append(r.date)
            if persistent and stable and not absent:
                embs = [app.embed(boat_crop(img(r.file), r.cx, r.cy, r.angle, r.len_px))
                        for r in m.itertuples()]
                sims = [app.sim(embs[i], embs[i + 1]) for i in range(len(embs) - 1)]
                appear = float(np.min(sims)) if sims else np.nan
            if not persistent:
                status = "not persistent"
            elif not stable:
                status = "same berth, boat changes (position/length/heading differ)"
            elif absent:
                status = "left and came back (in use)"
            elif not np.isnan(appear) and appear < cfg["appearance_thr"]:
                status = "same spot, appearance changed - check"
            else:
                status = "CANDIDATE abandoned"
            # neighbours within 30 m in the reference image (dense mooring rows)
            last_file = rows.file.iloc[-1]
            ld = dets[(dets.file == last_file)]
            neigh = int((np.hypot(ld.rx - t["x"], ld.ry - t["y"]) * mpp < 30).sum()) - (last_file in set(m.file))
            track_rows.append(dict(
                track_id=tid, site_id=site, review_priority=PRIORITY.get(status, 3), status=status,
                first_seen=m.date.iloc[0], last_seen=m.date.iloc[-1], years_seen=round(span, 1),
                n_detected=len(m), n_images_in_period=n_between, presence=round(presence, 2),
                still_present_latest=bool(last_file in set(m.file)),
                length_m=round(float(m.len_m.median()), 1), position_spread_m=round(spread_m, 1),
                length_change=round(len_change, 2), heading_change_deg=round(angle_change, 1),
                absent_dates=" ".join(absent),
                appearance_min_sim=round(appear, 3) if not np.isnan(appear) else None,
                neighbours_30m=max(neigh, 0), mean_conf=round(float(m.conf.mean()), 2),
                ref_x=round(float(t["x"]), 1), ref_y=round(float(t["y"]), 1),
                ref_file=rows.file.iloc[-1],
                # columns for manual review (Table 1 of Payne & Turner 2023)
                review_abandoned="", review_condition="", review_notes=""))
            if status in PRIORITY or (cfg.get("strips_all") and persistent):
                cand_n += 1
                strip_jobs.append((tid, t["x"], t["y"], m, track_rows[-1]))
        make_strips(out, site, rows, data, strip_jobs, mpp, img)
        img_cache.clear()
        make_overview(out, site, rows, data, [r for r in track_rows if r["site_id"] == site])
        log(f"track {site}: {len(tracks)} boats tracked, {cand_n} to review")

    tr = pd.DataFrame(track_rows)
    tr = tr.sort_values(["review_priority", "site_id", "years_seen"], ascending=[True, True, False])
    dets.drop(columns=["dx", "dy", "mpp"]).to_csv(out / "detections.csv", index=False)
    tr.to_csv(out / "tracks.csv", index=False)
    try:
        with pd.ExcelWriter(out / "tracks.xlsx") as xw:
            tr[tr.review_priority <= 2].to_excel(xw, sheet_name="To review", index=False)
            tr.to_excel(xw, sheet_name="All tracks", index=False)
            summary = tr.groupby("site_id").status.value_counts().unstack(fill_value=0)
            summary.to_excel(xw, sheet_name="Per site")
    except Exception as e:
        log(f"  (xlsx not written: {e})")
    log(f"done: {len(dets)} detections, {len(tr)} tracks, "
        f"{(tr.review_priority == 1).sum()} candidates + {(tr.review_priority == 2).sum()} to check "
        f"[appearance check: {app.method}]  -> {out}")


# ------------------------------------------------------------------ review images
def make_strips(out, site, rows, data, jobs, mpp, load, panel=260):
    """One strip per candidate: a panel per date centred on the boat's position, boat outlined
    in green where detected; a yellow cross marks the spot where it was not detected."""
    if not jobs:
        return
    panels = {j[0]: [] for j in jobs}
    for r in rows.itertuples():
        im = load(r.file)
        for tid, rx, ry, m, info in jobs:
            win = int(max(25 / mpp, m.len_px.max() * 2.2))  # at least 25 m wide
            x, y = rx - r.dx, ry - r.dy  # reference -> this image
            x0, y0 = int(x - win / 2), int(y - win / 2)
            crop = np.zeros((win, win, 3), np.uint8)
            sx0, sy0 = max(x0, 0), max(y0, 0)
            sx1, sy1 = min(x0 + win, im.shape[1]), min(y0 + win, im.shape[0])
            if sx1 > sx0 and sy1 > sy0:
                crop[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = im[sy0:sy1, sx0:sx1]
            k = panel / win
            crop = cv2.resize(crop, (panel, panel))
            hit = m[m.file == r.file]
            if len(hit):
                d = hit.iloc[0]
                box = cv2.boxPoints(((float((d.cx - x0) * k), float((d.cy - y0) * k)),
                                     (float(d.len_px * k), float(d.wid_px * k)), float(d.angle)))
                cv2.polylines(crop, [box.astype(np.int32)], True, (0, 255, 0), 2)
            else:
                c = panel // 2
                cv2.drawMarker(crop, (c, c), (0, 220, 255), cv2.MARKER_TILTED_CROSS, 18, 2)
            col = (60, 170, 60) if len(hit) else (70, 70, 200)
            crop = cv2.copyMakeBorder(crop, 4, 30, 4, 4, cv2.BORDER_CONSTANT, value=col)
            cv2.putText(crop, r.date + ("" if len(hit) else "  not detected"), (8, panel + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            panels[tid].append(crop)
    d = out / "review" / site
    d.mkdir(parents=True, exist_ok=True)
    for tid, rx, ry, m, info in jobs:
        strip = np.hstack(panels[tid])
        head = np.full((40, strip.shape[1], 3), 30, np.uint8)
        cv2.putText(head, f"{tid}  len ~{info['length_m']} m  seen {info['first_seen']} .. {info['last_seen']}"
                          f"  ({info['n_detected']}/{info['n_images_in_period']} images)  {info['status']}",
                    (10, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
        sub = d / {1: "1_candidates", 2: "2_check_appearance_changed"}.get(info["review_priority"], "3_other")
        sub.mkdir(exist_ok=True)
        cv2.imwrite(str(sub / f"{tid}.jpg"), np.vstack([head, strip]), [cv2.IMWRITE_JPEG_QUALITY, 88])


def make_overview(out, site, rows, data, trs):
    ref = rows.iloc[-1]
    im = cv2.imread(ref.path, cv2.IMREAD_REDUCED_COLOR_4)
    for r in trs:
        if r["status"] == "not persistent":
            continue
        p = (int(r["ref_x"] / 4), int(r["ref_y"] / 4))
        if r["review_priority"] <= 2:
            col = (0, 0, 255) if r["review_priority"] == 1 else (0, 165, 255)
            cv2.circle(im, p, 14, col, 3)
            cv2.putText(im, r["track_id"].split("-")[1], (p[0] + 16, p[1] + 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
        else:
            cv2.circle(im, p, 4, (0, 200, 255), -1)
    d = out / "review"
    d.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(d / f"{site}_overview.jpg"), im, [cv2.IMWRITE_JPEG_QUALITY, 85])


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("data", help="folder with the images")
    ap.add_argument("--out", default="results")
    ap.add_argument("--sites", nargs="*", help="only these site numbers, e.g. 01 17")
    ap.add_argument("--no-deep", action="store_true", help="skip DINOv2 appearance model")
    ap.add_argument("--strips-all", action="store_true", help="review strips for every persistent spot, not only candidates")
    for k, v in DEFAULTS.items():
        ap.add_argument("--" + k.replace("_", "-"), type=type(v), default=v)
    cfg = vars(ap.parse_args())
    run(cfg)

## 5. Quick test on one site (≈ 2–4 min)
Runs site 01 only, so you can check the output before the full run.
This also downloads the detector (~40 MB) and the appearance model on first use.

In [ ]:
!python boat_pipeline.py "$DATA_DIR" --out "$OUT_DIR" --sites 01

## 6. Full run – all 41 sites (≈ 20–40 min on a T4 GPU)
Already-finished images are skipped.

In [ ]:
!python boat_pipeline.py "$DATA_DIR" --out "$OUT_DIR"

## 7. Look at the results
In your results folder on Drive:
- **`tracks.xlsx`** – sheet *To review* lists every boat to check, with empty columns
  `review_abandoned` (yes/no), `review_condition` (intact / damaged / partially submerged / broken down) and `review_notes`.
- **`review/<site>/1_candidates/`** – boats in the same spot ≥ 5 years with stable position, size and look.
- **`review/<site>/2_check_appearance_changed/`** – same spot ≥ 5 years but the look changed: either a
  different boat in the same berth, **or a boat that is decaying** – check these too.
- **`review/<site>_overview.jpg`** – where the review boats are in each site (red = 1, orange = 2).

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
tr = pd.read_csv(f'{OUT_DIR}/tracks.csv')
print(tr.status.value_counts(), '\n')
display(tr[tr.status != 'not persistent'].groupby('site_id').status.value_counts().unstack(fill_value=0))

# all priority-1 candidates (boat outlined in green; yellow cross = not detected that year)
for p in sorted(glob.glob(f'{OUT_DIR}/review/*/1_candidates/*.jpg')):
    print(p.split('/')[-3], p.split('/')[-1]); display(Image(p, width=1400))

## 8. Automatic coordinates (≈ 10–30 min, depends on the map server)
Matches each site's coastline to OpenStreetMap to get exact coordinates — no manual control points needed.
Only sites marked **OK** get coordinates written; CHECK/FAILED sites keep their approximate coordinates.
Failed sites are retried once with a wider search (2.5 km). Map data is saved in `osm_cache`, so re-runs are fast.
If the map servers are very busy you'll see "map server busy ... trying another" — that's normal.
Sites where you already filled in your own control points are kept as they are.

Results:
- `coordinates_georef.xlsx` in the results folder — your coordinates file with the *Control points* sheet
  filled in, plus a sheet **Georef status** (OK / CHECK / FAILED per site).
- `georef_check/` — one image per site with the OpenStreetMap shoreline in red. **If the red line sits on the
  coast, the site is correct.** For CHECK/FAILED sites, look at the image; if it is wrong, measure that site
  by hand in the Control points sheet (description not starting with "auto") and run this step again.

In [ ]:
%%writefile georef_osm.py
#!/usr/bin/env python3
"""
georef_osm.py - automatic georeferencing of the Google Earth exports by matching
their coastline to OpenStreetMap (coastline, piers, breakwaters, water areas).

Each export is north-up at a known scale (scale bar), and the site coordinate in the
spreadsheet is within ~500 m of the image centre. So we only need to find the
shift (plus a small scale correction) that makes the image's water/land boundary
fall on the OSM shoreline.

Reads   DATA/                 images
        COORDS xlsx           sheet "Images" (site coordinates), optional "Control points"
Writes  OUT/coordinates_georef.xlsx   copy of COORDS with "Control points" filled in for
                                      every site that has no complete manual points,
                                      plus a sheet "Georef status"
        OUT/georef_check/<site>.jpg   image with the matched OSM lines drawn in red
        OUT/osm_cache/                downloaded OSM data (re-runs don't download again)

Map data © OpenStreetMap contributors (ODbL) - used here only to compute coordinates.
"""
import argparse
import json
import math
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

K = 111320.0                 # metres per degree latitude
SEARCH_M = 900               # search radius around the spreadsheet coordinate (sheet says up to ~500 m off)
CLIP_M = 20                  # distances beyond this count as "no match"
INLIER_M = 5
SCALES = [0.97, 0.98, 0.99, 1.0, 1.01, 1.02, 1.03]
OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.private.coffee/api/interpreter",
            "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter"]


# ------------------------------------------------------------------ image side
def read_scale_bar(img, bar_m=50.0):
    h, w = img.shape[:2]
    reg = img[h - 70:h, w - 900:w]
    white = np.all(reg > 215, axis=2)
    best = 0
    for row in white:
        r = np.diff(np.concatenate(([0], row.view(np.int8), [0])))
        s, e = np.where(r == 1)[0], np.where(r == -1)[0]
        if len(s):
            best = max(best, int((e - s).max()))
    return bar_m / best if 150 < best < 880 else None


def water_mask(bgr):
    """Main water body: blue-ish or smooth+dark pixels, largest connected region(s)."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, s, v = [c.astype(np.float32) for c in cv2.split(hsv)]
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    mu = cv2.blur(g, (9, 9))
    sd = np.sqrt(np.maximum(cv2.blur(g * g, (9, 9)) - mu * mu, 0))
    blue = (h >= 80) & (h <= 125) & (s > 40) & (v < 170)
    smooth_dark = (sd < 6) & (v < 110)
    m = (blue | smooth_dark).astype(np.uint8) * 255
    H, W = m.shape
    m[int(H * .955):, :int(W * .06)] = 0          # Google Earth logo
    m[int(H * .94):, int(W * .93):] = 0           # scale bar
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    keep = np.zeros_like(m)
    if n <= 1:
        return keep
    big = 1 + int(np.argmax(st[1:, 4]))
    for i in range(1, n):
        x, y, w, hh, a = st[i]
        border = x == 0 or y == 0 or x + w >= W or y + hh >= H
        if i == big or (border and a > 0.02 * m.size):
            keep[lab == i] = 255
    inv = cv2.bitwise_not(keep)
    n2, l2, s2, _ = cv2.connectedComponentsWithStats(inv)
    for i in range(1, n2):
        if s2[i, 4] < 0.0005 * m.size:
            keep[l2 == i] = 255                   # fill boats/buoys inside water
    return keep


def water_mask_strict(bgr):
    """Stricter variant: clearly blue, saturated, smooth water only - avoids spilling into
    shadows and dark parks, but misses grey/glare water. georef_site tries both."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, s, v = [c.astype(np.float32) for c in cv2.split(hsv)]
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    mu = cv2.blur(g, (15, 15))
    sd = np.sqrt(np.maximum(cv2.blur(g * g, (15, 15)) - mu * mu, 0))
    hs, ss, vs = cv2.blur(h, (7, 7)), cv2.blur(s, (7, 7)), cv2.blur(v, (7, 7))
    m = ((hs >= 78) & (hs <= 128) & (ss > 55) & (vs > 25) & (vs < 210) & (sd < 18)).astype(np.uint8) * 255
    H, W = m.shape
    m[int(H * .955):, :int(W * .06)] = 0
    m[int(H * .94):, int(W * .93):] = 0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    core = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    n, lab, st, _ = cv2.connectedComponentsWithStats(core)
    keep = np.zeros_like(m)
    if n <= 1:
        return keep
    big = st[1:, 4].max()
    for i in range(1, n):
        if st[i, 4] >= max(0.25 * big, 0.01 * m.size):
            keep[lab == i] = 255
    keep = cv2.bitwise_or(keep, cv2.dilate(keep, k) & m)
    inv = cv2.bitwise_not(keep)
    n2, l2, s2, _ = cv2.connectedComponentsWithStats(inv)
    for i in range(1, n2):
        if s2[i, 4] < 0.0005 * m.size:
            keep[l2 == i] = 255
    return keep


def boundary(mask):
    """Water/land edge pixels, excluding the image border."""
    e = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))
    e[:3, :] = e[-3:, :] = 0
    e[:, :3] = e[:, -3:] = 0
    return (e > 0).astype(np.uint8)


# ------------------------------------------------------------------ OSM side
def fetch_osm(lat0, lon0, half_w_m, half_h_m, cache):
    if cache.exists():
        return json.loads(cache.read_text())
    import requests
    dlat, dlon = half_h_m / K, half_w_m / (K * math.cos(math.radians(lat0)))
    bb = f"{lat0 - dlat},{lon0 - dlon},{lat0 + dlat},{lon0 + dlon}"
    q = f"""[out:json][timeout:60];
(
  way["natural"="coastline"]({bb});
  way["man_made"~"^(pier|breakwater|groyne|quay)$"]({bb});
  way["natural"="water"]({bb});  relation["natural"="water"]({bb});
  way["waterway"~"^(riverbank|dock)$"]({bb});
  way["water"]({bb}); relation["water"]({bb});
);
out geom;"""
    last = None
    for attempt in range(8):
        url = OVERPASS[attempt % len(OVERPASS)]
        try:
            r = requests.post(url, data={"data": q}, timeout=(10, 70),
                              headers={"User-Agent": "abandoned-boats-thesis/1.0"})
            if r.status_code == 200:
                lines = []
                for el in r.json().get("elements", []):
                    if el["type"] == "way" and "geometry" in el:
                        lines.append([(p["lat"], p["lon"]) for p in el["geometry"]])
                    elif el["type"] == "relation":
                        for mem in el.get("members", []):
                            if "geometry" in mem:
                                lines.append([(p["lat"], p["lon"]) for p in mem["geometry"]])
                cache.write_text(json.dumps(lines))
                time.sleep(2)  # be polite to the free Overpass service
                return lines
            last = f"HTTP {r.status_code}"
        except Exception as e:  # network hiccup
            last = str(e)
        print(f"   map server busy ({url.split('/')[2]}: {str(last)[:60]}), trying another...", flush=True)
        time.sleep(3)
    raise RuntimeError(f"OpenStreetMap download failed: {last}")


def rasterize(lines, lat0, lon0, res, cw, ch):
    """Draw OSM lines on a canvas centred on (lat0, lon0), res metres per pixel."""
    c = np.zeros((ch, cw), np.uint8)
    coslat = math.cos(math.radians(lat0))
    for ln in lines:
        pts = np.array([((lon - lon0) * K * coslat / res + cw / 2,
                         -(lat - lat0) * K / res + ch / 2) for lat, lon in ln], np.float32)
        if len(pts) >= 2:
            cv2.polylines(c, [np.round(pts).astype(np.int32)], False, 1, 1)
    return c


# ------------------------------------------------------------------ matching
MIN_OSM_PX_M = 300          # need at least this much OSM shoreline (metres) inside the image


def cost_maps(b, osm, dt_osm, res):
    """Symmetric chamfer cost for every shift of template b (image boundary) over the canvas.
    a = mean distance image edge -> OSM line, c = mean distance OSM line (inside frame) -> image edge."""
    bf = b.astype(np.float32)
    n = bf.sum()
    a = cv2.matchTemplate(dt_osm, bf, cv2.TM_CCORR) / max(n, 1)
    dt_img = np.minimum(cv2.distanceTransform((1 - b).astype(np.uint8), cv2.DIST_L2, 5) * res, CLIP_M)
    of = osm.astype(np.float32)
    cnt = cv2.matchTemplate(of, np.ones_like(bf), cv2.TM_CCORR)
    c = cv2.matchTemplate(of, dt_img.astype(np.float32), cv2.TM_CCORR) / np.maximum(cnt, 1)
    c[cnt * res < MIN_OSM_PX_M] = CLIP_M        # too little map inside the frame here
    return 0.35 * a + 0.65 * c, a, c


def search(b, osm, dt_osm, res, scales, window=None):
    best = None
    for s in scales:
        t = cv2.resize(b, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        if t.sum() < 200:
            continue
        if window is not None:
            u0, v0, r = window
            y0, x0 = max(v0 - r, 0), max(u0 - r, 0)
            sub_o = osm[y0:v0 + r + t.shape[0], x0:u0 + r + t.shape[1]]
            sub_d = dt_osm[y0:v0 + r + t.shape[0], x0:u0 + r + t.shape[1]]
            off = (x0, y0)
        else:
            sub_o, sub_d, off = osm, dt_osm, (0, 0)
        if sub_o.shape[0] < t.shape[0] or sub_o.shape[1] < t.shape[1]:
            continue
        cm, _, _ = cost_maps(t, sub_o, sub_d, res)
        v, u = np.unravel_index(np.argmin(cm), cm.shape)
        if best is None or cm[v, u] < best[0]:
            best = (float(cm[v, u]), s, int(u + off[0]), int(v + off[1]), cm)
    return best


def canvas(lines, lat0, lon0, res, half_w, half_h, search_m):
    cw = int(2 * (half_w * 1.04 + search_m) / res)
    ch = int(2 * (half_h * 1.04 + search_m) / res)
    osm = rasterize(lines, lat0, lon0, res, cw, ch)
    dt = cv2.distanceTransform((1 - osm).astype(np.uint8), cv2.DIST_L2, 5) * res
    return osm, np.minimum(dt, CLIP_M).astype(np.float32), cw, ch


def georef_site(img, lat0, lon0, mpp, cache, search_m=None, cache_wide=None):
    search_m = search_m or SEARCH_M
    small4 = cv2.resize(img, None, fx=1 / 4, fy=1 / 4, interpolation=cv2.INTER_AREA)
    H, W = img.shape[:2]
    half_w, half_h = W * mpp / 2, H * mpp / 2
    lines = fetch_osm(lat0, lon0, half_w + search_m + 300, half_h + search_m + 300, cache)
    if not lines:
        return dict(status="FAILED", reason="no OSM shoreline in this area")

    # try both water detections, keep the one that fits the map best
    masks = {"normal": water_mask(small4), "strict": water_mask_strict(small4)}
    res8, res4 = mpp * 8, mpp * 4
    osm8, dt8, cw8, ch8 = canvas(lines, lat0, lon0, res8, half_w, half_h, search_m)
    cands = []
    for name, wm in masks.items():
        if (wm > 0).mean() < 0.01:
            continue
        b4 = boundary(wm)
        b8 = (cv2.resize(b4, (W // 8, H // 8), interpolation=cv2.INTER_AREA) > 0).astype(np.uint8)
        best = search(b8, osm8, dt8, res8, SCALES)
        if best:
            cands.append((best[0], name, b4, best))
    if not cands:
        return dict(status="FAILED", reason="no water found in image")
    cands.sort(key=lambda c: c[0])
    _, mask_name, b4, (sc, s, u, v, cm) = cands[0]
    r = int(60 / res8)
    cm2 = cm.copy()
    cm2[max(v - r, 0):v + r, max(u - r, 0):u + r] = np.inf
    ambiguity = float(sc / max(cm2.min(), 1e-6))
    tl_e, tl_s = (u - cw8 / 2) * res8, (v - ch8 / 2) * res8

    # fine level
    osm4, dt4, cw4, ch4 = canvas(lines, lat0, lon0, res4, half_w, half_h, search_m)
    u0, v0 = int(tl_e / res4 + cw4 / 2), int(tl_s / res4 + ch4 / 2)
    best = search(b4, osm4, dt4, res4, [s - 0.005, s, s + 0.005], window=(u0, v0, 6))
    if best is None:
        return dict(status="FAILED", reason="fine search failed")
    sc, s, u, v, _ = best
    t = cv2.resize(b4, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
    patch = dt4[v:v + t.shape[0], u:u + t.shape[1]]
    inlier = float((patch[t > 0] <= INLIER_M).mean())
    osm_in = osm4[v:v + t.shape[0], u:u + t.shape[1]]
    dt_img = cv2.distanceTransform((1 - t).astype(np.uint8), cv2.DIST_L2, 5) * res4
    coverage = float((dt_img[osm_in > 0] <= INLIER_M).mean()) if osm_in.any() else 0.0
    osm_len_m = float(osm_in.sum() * res4)
    out = dict(score_m=sc, scale=s, inlier=inlier, coverage=coverage, ambiguity=ambiguity,
               mask=mask_name, osm_len_m=osm_len_m,
               tl_east_m=(u - cw4 / 2) * res4, tl_south_m=(v - ch4 / 2) * res4, lines=lines)
    out["m_per_px_full"] = s * mpp
    shift_e = out["tl_east_m"] + W / 2 * out["m_per_px_full"]
    shift_s = out["tl_south_m"] + H / 2 * out["m_per_px_full"]
    out["centre_offset_m"] = math.hypot(shift_e, shift_s)
    near_edge = out["centre_offset_m"] > 0.95 * search_m
    # thresholds calibrated by checking 18 real sites by eye (all OK sites were correct)
    ok = coverage >= 0.5 and inlier >= 0.15 and ambiguity <= 0.8 and osm_len_m >= 500
    check = coverage >= 0.35 and ambiguity < 0.97 and osm_len_m >= MIN_OSM_PX_M
    out["status"] = "OK" if ok else ("CHECK" if check else "FAILED")
    out["near_edge"] = near_edge
    out["reason"] = "" if ok else ("weak match - look at the check image" if check else "no reliable match")
    if cache_wide is not None and out["status"] != "OK" and search_m < 2500:
        # the sheet coordinate may be far off: search a much larger area once
        try:
            wide = georef_site(img, lat0, lon0, mpp, cache_wide, search_m=2500)
        except Exception as e:
            wide = dict(status="FAILED", reason=str(e)[:120])
        if wide.get("status") == "OK":
            wide["reason"] = (wide.get("reason") or "") + " (found with wide search)"
            return wide
    return out


def pixel_to_latlon(r, x, y, lat0, lon0):
    e = r["tl_east_m"] + x * r["m_per_px_full"]
    s = r["tl_south_m"] + y * r["m_per_px_full"]
    return lat0 - s / K, lon0 + e / (K * math.cos(math.radians(lat0)))


def check_image(img, r, lat0, lon0, path, title):
    sm = cv2.resize(img, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_AREA)
    coslat = math.cos(math.radians(lat0))
    f = r["m_per_px_full"] * 4
    for ln in r["lines"]:
        pts = []
        for lat, lon in ln:
            e = (lon - lon0) * K * coslat - r["tl_east_m"]
            s = -(lat - lat0) * K - r["tl_south_m"]
            pts.append((e / f, s / f))
        if len(pts) >= 2:
            cv2.polylines(sm, [np.round(np.array(pts)).astype(np.int32)], False, (0, 0, 255), 2, cv2.LINE_AA)
    bar = np.full((44, sm.shape[1], 3), 25, np.uint8)
    cv2.putText(bar, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imwrite(str(path), np.vstack([sm, bar]), [cv2.IMWRITE_JPEG_QUALITY, 85])


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data"), ap.add_argument("coords")
    ap.add_argument("--out", default="results")
    ap.add_argument("--sites", nargs="*")
    ap.add_argument("--no-wide", action="store_true", help="don't retry failed sites with a 2.5 km search")
    a = ap.parse_args()
    data, out = Path(a.data), Path(a.out)
    (out / "georef_check").mkdir(parents=True, exist_ok=True)
    (out / "osm_cache").mkdir(parents=True, exist_ok=True)

    images = pd.read_excel(a.coords, sheet_name="Images", dtype={"Site no.": str})
    images["Site no."] = images["Site no."].str.strip()
    sheets = pd.read_excel(a.coords, sheet_name=None, dtype={"Site no.": str})
    manual = set()
    if "Control points" in sheets:
        cp = sheets["Control points"]
        cp["Site no."] = cp["Site no."].astype(str).str.strip().str.zfill(2)
        full = cp.dropna(subset=["Pixel X", "Pixel Y", "Latitude", "Longitude"])
        full = full[~full["Description"].astype(str).str.startswith("auto")]
        manual = {s for s, g in full.groupby("Site no.") if len(g) >= 3}

    results = []
    for site_no, g in images.groupby("Site no.", sort=True):
        if a.sites and site_no not in a.sites:
            continue
        g = g.sort_values("Image date")
        ref = g.iloc[-1]
        name = ref["Site"]
        if site_no in manual:
            results.append(dict(site_no=site_no, site=name, ref_file=ref["File"], status="MANUAL",
                                reason="your own control points are kept"))
            print(f"{site_no} {name}: manual control points present - skipped")
            continue
        lat0, lon0 = float(ref["Latitude (approx.)"]), float(ref["Longitude (approx.)"])
        img = cv2.imread(str(data / ref["File"]))
        if img is None:
            results.append(dict(site_no=site_no, site=name, ref_file=ref["File"], status="FAILED",
                                reason="image not found"))
            continue
        mpp = read_scale_bar(img) or 0.143
        t0 = time.time()
        try:
            r = georef_site(img, lat0, lon0, mpp, out / "osm_cache" / f"{site_no}.json",
                            cache_wide=None if a.no_wide else out / "osm_cache" / f"{site_no}_wide.json")
        except Exception as e:
            r = dict(status="FAILED", reason=str(e)[:200])
        row = dict(site_no=site_no, site=name, ref_file=ref["File"], status=r["status"],
                   reason=r.get("reason", "") + ("" if r["status"] == "OK" else
                                                 " - coordinates NOT written (approximate kept)"))
        if "tl_east_m" in r:
            H, W = img.shape[:2]
            pts = [(0.2 * W, 0.2 * H), (0.8 * W, 0.3 * H), (0.5 * W, 0.85 * H)]
            row.update(points=[(x, y) + pixel_to_latlon(r, x, y, lat0, lon0) for x, y in pts],
                       inlier=round(r["inlier"], 2), coverage=round(r["coverage"], 2),
                       ambiguity=round(r["ambiguity"], 2), scale=round(r["scale"], 3),
                       moved_from_sheet_m=round(r["centre_offset_m"]))
            title = (f"{site_no} {name}: {r['status']}  match {r['inlier']:.0%}  "
                     f"OSM covered {r['coverage']:.0%}  (red = OpenStreetMap shoreline)")
            check_image(img, r, lat0, lon0, out / "georef_check" / f"{site_no}_{name}.jpg", title)
        results.append(row)
        print(f"{site_no} {name}: {r['status']} {r.get('reason', '')} "
              f"inlier={r.get('inlier', 0):.2f} cover={r.get('coverage', 0):.2f} "
              f"amb={r.get('ambiguity', 0):.2f} moved={r.get('centre_offset_m', 0):.0f} m "
              f"({time.time() - t0:.0f}s)", flush=True)

    # ---- write coordinates_georef.xlsx (copy of the input, Control points filled)
    import openpyxl
    wb = openpyxl.load_workbook(a.coords)
    hdr = ["Site no.", "Site", "Reference file", "Point no.", "Pixel X", "Pixel Y",
           "Latitude", "Longitude", "Description"]
    if "Control points" not in wb.sheetnames:
        ws = wb.create_sheet("Control points")
        ws.append(hdr)
    ws = wb["Control points"]
    col = {c.value: i for i, c in enumerate(ws[1])}
    for res in results:
        if "points" not in res or res["status"] != "OK":
            continue                       # CHECK/FAILED keep their approximate coordinates
        rows = [r for r in ws.iter_rows(min_row=2) if str(r[col["Site no."]].value).strip().zfill(2) == res["site_no"]]
        while len(rows) < 3:
            ws.append([res["site_no"], res["site"], res["ref_file"], len(rows) + 1])
            rows.append(list(ws.iter_rows(min_row=ws.max_row, max_row=ws.max_row))[0])
        for k, (r, (x, y, lat, lon)) in enumerate(zip(rows[:3], res["points"]), 1):
            r[col["Reference file"]].value = res["ref_file"]
            r[col["Point no."]].value = k
            r[col["Pixel X"]].value = int(round(x))
            r[col["Pixel Y"]].value = int(round(y))
            r[col["Latitude"]].value = round(lat, 7)
            r[col["Longitude"]].value = round(lon, 7)
            r[col["Description"]].value = f"auto OSM match ({res['status']})"
    if "Georef status" in wb.sheetnames:
        del wb["Georef status"]
    st = wb.create_sheet("Georef status")
    keys = ["site_no", "site", "status", "reason", "inlier", "coverage", "ambiguity", "scale",
            "moved_from_sheet_m", "ref_file"]
    st.append(["Site no.", "Site", "Status", "Reason", "Shoreline matched", "OSM covered",
               "Ambiguity (lower=better)", "Scale factor", "Moved from sheet coordinate (m)", "Reference file"])
    for res in results:
        st.append([res.get(k) for k in keys])
    dst = out / "coordinates_georef.xlsx"
    wb.save(dst)
    s = pd.Series([r["status"] for r in results]).value_counts().to_dict()
    print(f"\n{s}\n-> {dst}\n-> check images in {out / 'georef_check'}")


if __name__ == "__main__":
    main()

In [ ]:
COORDS = '/content/drive/MyDrive/Abandoned_ships___image_coordinates.xlsx'   # your coordinates file
!python georef_osm.py "$DATA_DIR" "$COORDS" --out "$OUT_DIR"

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display
st = pd.read_excel(f'{OUT_DIR}/coordinates_georef.xlsx', sheet_name='Georef status')
display(st)
for p in sorted(glob.glob(f'{OUT_DIR}/georef_check/*.jpg'))[:6]:
    display(Image(p, width=1000))

## 8b. Align to the website basemap (recommended, ≈ 10–20 min)
Matches every site's image directly against the satellite basemap the website shows
(Esri World Imagery), using piers, roads and buildings — works where step 8 failed (glare, creeks).

**Before running:** upload the website's current `points.json` into the `abandoned ships results` folder.

Output in the results folder:
- `points_new.json` — **the file for the website**: rename to `points.json` and put it in your repository.
  Only sites marked OK are moved; others keep their current coordinates.
- `esri_check/` — one image per site: your image (magenta) over the basemap (green); **grey = they line up**.
- `esri_transforms.json` — the per-site result (OK / CHECK / FAILED).

In [ ]:
%%writefile georef_esri.py
#!/usr/bin/env python3
"""
georef_esri.py - place every site's reference image on the website's own basemap
(Esri World Imagery) by image-to-image matching. No manual work, no shoreline needed.

For each site:
  1. download basemap tiles around the spreadsheet coordinate (zoom 16, +-SEARCH_M)
  2. find where the image fits (edge-image correlation, a few scale steps)
  3. refine at zoom 18 (~0.45 m/px)
Writes  OUT/esri_transforms.json   per-site transform image-pixel -> lat/lon
        OUT/esri_check/<site>.jpg  basemap (green) vs your image (magenta) at the match:
                                   grey where they agree = correct
        OUT/esri_cache/            downloaded tiles (re-runs are fast)
Imagery: Esri, Maxar, Earthstar Geographics (used here only to compute positions).
"""
import argparse, json, math, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

TILE_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
SEARCH_M = 2600
SCALES = [0.94, 0.96, 0.98, 1.0, 1.02, 1.04, 1.06]
R_EARTH_PX = 156543.03392804097          # metres per pixel at zoom 0, equator


# ---------------------------------------------------------------- web mercator
def ll2px(lat, lon, z):
    n = 256 * 2 ** z
    x = (lon + 180) / 360 * n
    y = (1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n
    return x, y


def px2ll(x, y, z):
    n = 256 * 2 ** z
    lon = x / n * 360 - 180
    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
    return lat, lon


def mpp_at(lat, z):
    return R_EARTH_PX * math.cos(math.radians(lat)) / 2 ** z


# ---------------------------------------------------------------- tiles
class Tiles:
    def __init__(self, cache, fetch=None):
        self.cache = Path(cache)
        self.fetch = fetch or self._http
        self.session = None

    def _http(self, z, x, y):
        import requests
        if self.session is None:
            self.session = requests.Session()
            self.session.headers["User-Agent"] = "abandoned-boats-thesis/1.0"
        for attempt in range(4):
            try:
                r = self.session.get(TILE_URL.format(z=z, x=x, y=y), timeout=20)
                if r.status_code == 200:
                    return r.content
            except Exception:
                pass
            time.sleep(1 + attempt)
        return None

    def get(self, z, x, y):
        p = self.cache / str(z) / f"{x}_{y}.jpg"
        if p.exists():
            return cv2.imdecode(np.frombuffer(p.read_bytes(), np.uint8), cv2.IMREAD_COLOR)
        data = self.fetch(z, x, y)
        if not data:
            return None
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(data)
        return cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)

    def mosaic(self, z, px0, py0, px1, py1):
        """basemap image covering global pixel box [px0,px1) x [py0,py1) at zoom z"""
        tx0, ty0, tx1, ty1 = int(px0 // 256), int(py0 // 256), int(px1 // 256), int(py1 // 256)
        coords = [(x, y) for y in range(ty0, ty1 + 1) for x in range(tx0, tx1 + 1)]
        with ThreadPoolExecutor(8) as ex:
            imgs = list(ex.map(lambda c: self.get(z, *c), coords))
        W, H = (tx1 - tx0 + 1) * 256, (ty1 - ty0 + 1) * 256
        m = np.zeros((H, W, 3), np.uint8)
        missing = 0
        for (x, y), im in zip(coords, imgs):
            if im is None:
                missing += 1
                continue
            m[(y - ty0) * 256:(y - ty0 + 1) * 256, (x - tx0) * 256:(x - tx0 + 1) * 256] = im[:256, :256]
        ox, oy = int(px0) - tx0 * 256, int(py0) - ty0 * 256
        return m[oy:oy + int(py1 - py0), ox:ox + int(px1 - px0)], missing / max(len(coords), 1)


# ---------------------------------------------------------------- matching
def orient(bgr, sigma):
    """edge-orientation field (doubled angle, so dark-on-light and light-on-dark match),
    weighted by a saturating edge strength: random water ripples average out,
    coastlines, piers, streets and buildings line up"""
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    g = cv2.GaussianBlur(g, (0, 0), sigma)
    gx, gy = cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)
    mag = np.hypot(gx, gy)
    c = float(np.median(mag)) + 1e-3
    w = mag / (mag + 2 * c)
    ang2 = 2 * np.arctan2(gy, gx)
    return (w * np.cos(ang2)).astype(np.float32), (w * np.sin(ang2)).astype(np.float32)


def mask_overlays(fields):
    out = []
    for e in fields:
        e = e.copy()
        h, w = e.shape
        e[int(h * .955):, :int(w * .06)] = 0
        e[int(h * .94):, int(w * .93):] = 0
        out.append(e)
    return out


def best_match(base_f, tmpl_f, win=None):
    """orientation correlation for every shift; returns (score, u, v, ratio second/best)"""
    (br, bi), (tr, ti) = base_f, tmpl_f
    if win is not None:
        u0, v0, r = win
        y0, x0 = max(v0 - r, 0), max(u0 - r, 0)
        sl = (slice(y0, v0 + r + tr.shape[0]), slice(x0, u0 + r + tr.shape[1]))
        br, bi = br[sl], bi[sl]
    else:
        x0 = y0 = 0
    if br.shape[0] < tr.shape[0] or br.shape[1] < tr.shape[1]:
        return None
    num = cv2.matchTemplate(br, tr, cv2.TM_CCORR) + cv2.matchTemplate(bi, ti, cv2.TM_CCORR)
    tw = np.sqrt(tr * tr + ti * ti)
    tnorm = float(np.sqrt((tr * tr + ti * ti).sum())) + 1e-6
    bmag2 = br * br + bi * bi
    bnorm = np.sqrt(np.maximum(cv2.matchTemplate(bmag2, (tw > 0).astype(np.float32), cv2.TM_CCORR), 0)) + 1e-6
    sm = num / (tnorm * bnorm)
    # featureless basemap (open sea, missing tiles) must not win
    cover = cv2.matchTemplate(bmag2, np.ones_like(tr), cv2.TM_CCORR) / tr.size
    sm[(cover < 0.15 * float((tr * tr + ti * ti).mean())) | ~np.isfinite(sm)] = -1
    v, u = np.unravel_index(np.argmax(sm), sm.shape)
    best = float(sm[v, u])
    r = max(8, tr.shape[1] // 25)
    sm2 = sm.copy()
    sm2[max(v - r, 0):v + r, max(u - r, 0):u + r] = -1
    second = float(sm2.max()) if sm2.size else -1
    return best, int(u + x0), int(v + y0), second / max(best, 1e-6)


def georef_site(img, lat0, lon0, mpp, tiles, search_m=SEARCH_M):
    H, W = img.shape[:2]
    # ---------- coarse, zoom 16
    z = 16
    r = mpp_at(lat0, z)
    cx, cy = ll2px(lat0, lon0, z)
    half_w = (W * mpp / 2 + search_m) / r
    half_h = (H * mpp / 2 + search_m) / r
    base, miss = tiles.mosaic(z, cx - half_w, cy - half_h, cx + half_w, cy + half_h)
    if miss > 0.3:
        return dict(status="FAILED", reason=f"{miss:.0%} of basemap tiles missing")
    base_e = orient(base, 1.0)
    cand = None
    for s in SCALES:
        f = mpp * s / r
        t = cv2.resize(img, (max(int(W * f), 8), max(int(H * f), 8)), interpolation=cv2.INTER_AREA)
        res = best_match(base_e, mask_overlays(orient(t, 1.0)))
        if res and (cand is None or res[0] > cand[0]):
            cand = res + (s,)
    if cand is None:
        return dict(status="FAILED", reason="coarse match failed")
    score16, u, v, ratio16, s = cand
    # top-left of image in global z16 px -> metres offset kept via lat/lon
    tl_lat, tl_lon = px2ll(cx - half_w + u, cy - half_h + v, z)

    # ---------- fine, zoom 18, around the coarse result
    z = 18
    r = mpp_at(lat0, z)
    tx, ty = ll2px(tl_lat, tl_lon, z)
    margin = 60 / r
    fw, fh = W * mpp * 1.05 / r, H * mpp * 1.05 / r
    base, miss = tiles.mosaic(z, tx - margin, ty - margin, tx + fw + margin, ty + fh + margin)
    if miss > 0.3:
        return dict(status="FAILED", reason=f"{miss:.0%} of zoom-18 tiles missing")
    base_e = orient(base, 1.5)
    fine = None
    for s2 in np.arange(s - 0.025, s + 0.0251, 0.005):
        f = mpp * s2 / r
        t = cv2.resize(img, (int(W * f), int(H * f)), interpolation=cv2.INTER_AREA)
        res = best_match(base_e, mask_overlays(orient(t, 1.5)), win=(int(margin), int(margin), int(margin)))
        if res and (fine is None or res[0] > fine[0]):
            fine = res + (s2,)
    if fine is None:
        return dict(status="FAILED", reason="fine match failed")
    score18, u, v, ratio18, s2 = fine
    s2 = float(s2)
    k = mpp * s2 / r                                     # global z18 px per image px
    u0, v0 = tx - margin + u, ty - margin + v           # global z18 px of image top-left
    # calibrated on test sites: correct matches had ratio 0.1-0.25 (best peak far above the rest)
    ok = score16 >= 0.07 and ratio16 <= 0.6 and score18 >= 0.04
    check = score16 >= 0.05 and ratio16 <= 0.85
    status = "OK" if ok else ("CHECK" if check else "FAILED")
    clat, clon = px2ll(u0 + W / 2 * k, v0 + H / 2 * k, 18)
    moved = math.hypot((clat - lat0) * 111320, (clon - lon0) * 111320 * math.cos(math.radians(lat0)))
    crop = base[int(v):int(v) + int(H * k), int(u):int(u) + int(W * k)]
    return dict(status=status, z=18, u0=u0, v0=v0, k=k, scale=s2, score16=score16, ratio16=ratio16,
                score18=score18, moved_m=moved, _crop=crop)


def check_image(img, crop, path, title):
    h, w = crop.shape[:2]
    if h < 10 or w < 10:
        return
    a = cv2.cvtColor(cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    b = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    a = cv2.equalizeHist(a); b = cv2.equalizeHist(b)
    ov = np.dstack([a, b, a])                           # magenta = your image, green = basemap
    s = 1400 / w
    ov = cv2.resize(ov, (1400, int(h * s)))
    side = np.hstack([cv2.resize(cv2.resize(img, (w, h)), (700, int(h * 700 / w))),
                      cv2.resize(crop, (700, int(h * 700 / w)))])
    bar = np.full((44, 1400, 3), 25, np.uint8)
    cv2.putText(bar, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imwrite(str(path), np.vstack([ov, side, bar]), [cv2.IMWRITE_JPEG_QUALITY, 82])


def read_scale_bar(img, bar_m=50.0):
    h, w = img.shape[:2]
    reg = img[h - 70:h, w - 900:w]
    white = np.all(reg > 215, axis=2)
    best = 0
    for row in white:
        rr = np.diff(np.concatenate(([0], row.view(np.int8), [0])))
        st, en = np.where(rr == 1)[0], np.where(rr == -1)[0]
        if len(st):
            best = max(best, int((en - st).max()))
    return bar_m / best if 150 < best < 880 else None


def update_points(points_in, tracks_csv, transforms, points_out):
    """recompute vessel coordinates for every site matched OK; keep the others as they are"""
    pts = json.loads(Path(points_in).read_text())
    tr = pd.read_csv(tracks_csv).set_index("track_id")
    n_upd, moved = 0, []
    for f in pts["features"]:
        p = f["properties"]
        if p.get("track") not in tr.index:
            continue
        t = tr.loc[p["track"]]
        site = t.site_id.split("_")[0]
        x = transforms.get(site)
        if not x or x.get("status") != "OK":
            continue
        lat, lon = px2ll(x["u0"] + t.ref_x * x["k"], x["v0"] + t.ref_y * x["k"], x["z"])
        olon, olat = f["geometry"]["coordinates"]
        moved.append(math.hypot((lat - olat) * 111320, (lon - olon) * 111320 * math.cos(math.radians(lat))))
        f["geometry"]["coordinates"] = [round(lon, 7), round(lat, 7)]
        p["approx"] = False
        p["accuracy"] = "matched to the basemap imagery (~1-5 m)"
        n_upd += 1
    Path(points_out).write_text(json.dumps(pts, ensure_ascii=False, separators=(",", ":")))
    left = sum(bool(f["properties"].get("approx")) for f in pts["features"])
    med = float(np.median(moved)) if moved else 0
    print(f"points updated: {n_upd} (median shift {med:.0f} m), still approximate: {left}  -> {points_out}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data"), ap.add_argument("coords")
    ap.add_argument("--out", default="results")
    ap.add_argument("--sites", nargs="*")
    ap.add_argument("--points", help="current points.json of the website: writes points_new.json next to the results")
    a = ap.parse_args()
    out = Path(a.out)
    (out / "esri_check").mkdir(parents=True, exist_ok=True)
    tiles = Tiles(out / "esri_cache")
    im = pd.read_excel(a.coords, sheet_name="Images", dtype={"Site no.": str})
    im["Site no."] = im["Site no."].str.strip()
    tf_path = out / "esri_transforms.json"
    result = json.loads(tf_path.read_text()) if tf_path.exists() else {}
    for site_no, g in im.groupby("Site no.", sort=True):
        if a.sites and site_no not in a.sites:
            continue
        ref = g.sort_values("Image date").iloc[-1]
        img = cv2.imread(str(Path(a.data) / ref["File"]))
        if img is None:
            print(f"{site_no}: image not found"); continue
        mpp = read_scale_bar(img) or 0.143
        lat0, lon0 = float(ref["Latitude (approx.)"]), float(ref["Longitude (approx.)"])
        t0 = time.time()
        try:
            r = georef_site(img, lat0, lon0, mpp, tiles)
        except Exception as e:
            r = dict(status="FAILED", reason=str(e)[:150])
        crop = r.pop("_crop", None)
        r.update(site=ref["Site"], ref_file=ref["File"], mpp=mpp)
        result[site_no] = r
        if crop is not None:
            check_image(img, crop, out / "esri_check" / f"{site_no}_{ref['Site']}.jpg",
                        f"{site_no} {ref['Site']}: {r['status']}  score {r.get('score16', 0):.2f}/{r.get('score18', 0):.2f}"
                        f"  moved {r.get('moved_m', 0):.0f} m  (grey = matches, magenta/green = differs)")
        print(f"{site_no} {ref['Site']}: {r['status']} {r.get('reason', '')} score16={r.get('score16', 0):.2f} "
              f"ratio={r.get('ratio16', 0):.2f} score18={r.get('score18', 0):.2f} moved={r.get('moved_m', 0):.0f} m "
              f"({time.time() - t0:.0f}s)", flush=True)
        tf_path.write_text(json.dumps(result, indent=1, ensure_ascii=False))
    print(pd.Series([v["status"] for v in result.values()]).value_counts().to_dict())
    print("->", tf_path, "and", out / "esri_check")
    if a.points:
        update_points(a.points, out / "tracks.csv", result, out / "points_new.json")


if __name__ == "__main__":
    main()

In [ ]:
COORDS = '/content/drive/MyDrive/Abandoned_ships___image_coordinates.xlsx'   # your coordinates file
POINTS = f'{OUT_DIR}/points.json'                                            # the website's current points.json
!pip -q install requests
!python georef_esri.py "$DATA_DIR" "$COORDS" --out "$OUT_DIR" --points "$POINTS"

## 9. Review, then make the website files
*(To skip the review for now, set `NO_REVIEW = 1` or `2` in the last cell and run it.)*

1. Download `tracks.xlsx` from the results folder and open it in Excel.
2. In the sheet **To review**, look at each boat's strip in `review/<site>/...` and fill in
   `review_abandoned` (**yes** / no), `review_condition` and `review_notes`.
3. Save it as **`tracks_reviewed.xlsx`** and upload it into the results folder
   (a new name, so re-running the pipeline never overwrites your review).
4. Run the two cells below (they use the coordinates from step 8).

Output in `website/`: an `images` folder (current view + raw/aligned comparison per year, captions
on the images) and `points.xlsx` with ID, Name, Location, coordinates, Vessel, Material, On site, Notes.
Fill in the vessel type and material in `points.xlsx`. When you later add the **Control points**
sheet to the coordinates file, run step 8 again to get exact coordinates.

In [ ]:
%%writefile export_website.py
#!/usr/bin/env python3
"""
export_website.py - make the website files for every boat you confirmed as abandoned.

Reads   RESULTS/tracks_reviewed.xlsx  (your reviewed copy of tracks.xlsx: rows with review_abandoned = yes)
        RESULTS/alignment.csv, RESULTS/detections.csv   (from boat_pipeline.py)
        COORDS xlsx              (sheet "Images": site centre coordinates;
                                  optional sheet "Control points": exact georeferencing)
Writes  WEBSITE/images/<ID>_<date>_raw.jpg / _aligned.jpg / _current.jpg
        WEBSITE/points.xlsx      one row per vessel, columns for the website panel

Usage:
  python export_website.py "abandoned ships" "abandoned ships results" coordinates.xlsx --out website
"""
import argparse
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# District per site (pre-filled - please check)
DISTRICTS = {
    "Silivri": "Silivri", "Selimpasa": "Silivri", "Mimarsinan": "Büyükçekmece",
    "Buyukcekmece": "Büyükçekmece", "Ambarli": "Avcılar", "Menekse": "Küçükçekmece",
    "Yesilkoy": "Bakırköy", "Bakirkoy": "Bakırköy", "Samatya": "Fatih", "Zeytinburnu": "Zeytinburnu",
    "Yenikapi": "Fatih", "Kumkapi": "Fatih", "Fener-Ayakapi": "Fatih", "Kabatas": "Beyoğlu",
    "Salacak-Harem": "Üsküdar", "Kurbagalidere": "Kadıköy", "Bostanci": "Kadıköy",
    "Kucukyali": "Maltepe", "Kartal": "Kartal", "Pendik": "Pendik", "Tuzla": "Tuzla",
    "Buyukada": "Adalar", "Heybeliada": "Adalar", "Burgazada": "Adalar", "Kinaliada": "Adalar",
    "Sivriada": "Adalar", "Goksu-Anadoluhisari": "Beykoz", "Cubuklu": "Beykoz", "Beykoz": "Beykoz",
    "AnadoluKavagi": "Beykoz", "Poyrazkoy": "Beykoz", "Istinye": "Sarıyer", "Yenikoy": "Sarıyer",
    "Kirecburnu": "Sarıyer", "Sariyer": "Sarıyer", "RumeliKavagi": "Sarıyer",
    "RumeliFeneri": "Sarıyer", "Karaburun": "Arnavutköy", "Sile": "Şile", "Agva": "Şile",
}

W_OUT, H_OUT = 1500, 1000   # crop size in original pixels (~210 x 140 m at 0.14 m/px)
CAP_H = 56


def yes(v):
    return str(v).strip().lower() in ("yes", "y", "ja", "evet", "1", "true", "x")


# ------------------------------------------------------------------ coordinates
def load_georef(coords_xlsx, align):
    """Per site: function (x_ref, y_ref) -> (lat, lon, accuracy_note)."""
    xl = pd.read_excel(coords_xlsx, sheet_name=None, dtype={"Site no.": str})
    images = xl.get("Images", next(iter(xl.values())))
    images["Site no."] = images["Site no."].astype(str).str.strip()
    cps = xl.get("Control points")
    funcs = {}
    names = dict(zip(images["Site no."], images["Site"]))  # proper Turkish spelling
    for site_no, g in images.groupby("Site no."):
        lat0, lon0 = float(g["Latitude (approx.)"].iloc[0]), float(g["Longitude (approx.)"].iloc[0])
        funcs[site_no] = ("centre", lat0, lon0)
    if cps is not None:
        cps = cps.dropna(subset=["Pixel X", "Pixel Y", "Latitude", "Longitude"])
        cps["Site no."] = cps["Site no."].astype(str).str.strip().str.zfill(2)
        for site_no, g in cps.groupby("Site no."):
            if len(g) < 3:
                continue
            # control-point pixels -> reference frame (control points may be on any year's image)
            px = []
            for _, r in g.iterrows():
                f = r.get("Reference file")
                dx, dy = (align.loc[f, "dx"], align.loc[f, "dy"]) if f in align.index else (0.0, 0.0)
                px.append((float(r["Pixel X"]) + dx, float(r["Pixel Y"]) + dy))
            A = np.c_[np.array(px), np.ones(len(px))]
            coef, *_ = np.linalg.lstsq(A, g[["Latitude", "Longitude"]].to_numpy(float), rcond=None)
            res = np.abs(A @ coef - g[["Latitude", "Longitude"]].to_numpy(float)).max(0)
            err_m = float(max(res[0] * 111320, res[1] * 111320 * math.cos(math.radians(g.Latitude.mean()))))
            auto = g["Description"].astype(str).str.startswith("auto").all() if "Description" in g else False
            funcs[site_no] = ("cp", coef, err_m, auto)
    return funcs, names


def to_latlon(geo, x, y, w, h, mpp):
    if geo[0] == "cp":
        lat, lon = np.array([x, y, 1.0]) @ geo[1]
        if geo[3]:
            return lat, lon, "automatic (matched to OpenStreetMap shoreline, ~5-15 m)"
        return lat, lon, f"control points (fit ±{max(geo[2], 2):.0f} m)"
    _, lat0, lon0 = geo
    lat = lat0 - (y - h / 2) * mpp / 111320
    lon = lon0 + (x - w / 2) * mpp / (111320 * math.cos(math.radians(lat0)))
    return lat, lon, "approximate: site centre, up to ~500 m off - replace with control points"


# ------------------------------------------------------------------ images
def crop(img, cx, cy):
    h, w = img.shape[:2]
    x0, y0 = int(cx - W_OUT / 2), int(cy - H_OUT / 2)
    out = np.zeros((H_OUT, W_OUT) + img.shape[2:], img.dtype)
    sx0, sy0, sx1, sy1 = max(x0, 0), max(y0, 0), min(x0 + W_OUT, w), min(y0 + H_OUT, h)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = img[sy0:sy1, sx0:sx1]
    return out


def caption(img, text, sub=""):
    bar = np.full((CAP_H, img.shape[1], 3), 25, np.uint8)
    cv2.putText(bar, text, (14, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
    if sub:
        cv2.putText(bar, sub, (14, 47), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)
    return np.vstack([img, bar])


def save(path, img, quality, width):
    if width and img.shape[1] > width:
        img = cv2.resize(img, (width, int(img.shape[0] * width / img.shape[1])), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(path), img, [cv2.IMWRITE_JPEG_QUALITY, quality])


def mark(img, r=46):
    c = (W_OUT // 2, H_OUT // 2)
    cv2.circle(img, c, r, (255, 255, 255), 5, cv2.LINE_AA)
    cv2.circle(img, c, r, (0, 0, 255), 2, cv2.LINE_AA)
    return img


def overlay(ref_grey, mov_grey):
    """Magenta = this year, green = reference year; grey where they agree."""
    return np.dstack([mov_grey, ref_grey, mov_grey])


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data"), ap.add_argument("results"), ap.add_argument("coords")
    ap.add_argument("--out", default="website")
    ap.add_argument("--reviewed", default=None, help="reviewed tracks file (default RESULTS/tracks.xlsx)")
    ap.add_argument("--quality", type=int, default=75, help="JPEG quality (lower = smaller files)")
    ap.add_argument("--width", type=int, default=0, help="resize images to this width in px (0 = keep 1500)")
    ap.add_argument("--aligned-only", action="store_true", help="skip the raw images (halves the image count)")
    ap.add_argument("--auto", type=int, choices=[0, 1, 2], default=0,
                    help="skip the review: export priority-1 boats (1) or priority 1+2 boats (2), labelled unverified")
    a = ap.parse_args()
    data, res, out = Path(a.data), Path(a.results), Path(a.out)
    (out / "images").mkdir(parents=True, exist_ok=True)

    # accept "yes" from any sheet (you will usually fill in "To review")
    src = Path(a.reviewed) if a.reviewed else (res / "tracks_reviewed.xlsx" if (res / "tracks_reviewed.xlsx").exists()
                                                else res / "tracks.xlsx")
    print(f"reading reviews from {src.name}")
    sheets = pd.read_excel(src, sheet_name=None)
    if a.auto:
        tr = sheets["All tracks"]
        tr = tr[tr.review_priority <= a.auto].copy()
        print(f"no review: exporting {len(tr)} boats with review priority <= {a.auto}")
    else:
        tr = pd.concat([d[d.review_abandoned.apply(yes)] for d in sheets.values()
                        if {"track_id", "review_abandoned"} <= set(d.columns)])
    tr = tr.drop_duplicates("track_id")
    if tr.empty:
        raise SystemExit("No boats to export (no review_abandoned = yes, or no candidates for --auto).")
    align = pd.read_csv(res / "alignment.csv").set_index("file")
    dets = pd.read_csv(res / "detections.csv")
    geo, site_names = load_georef(a.coords, align)

    rows, n_in_site = [], {}
    cache = {}

    def load(f, grey=False):
        key = (f, grey)
        if key not in cache:
            cache[key] = cv2.imread(str(data / f), cv2.IMREAD_GRAYSCALE if grey else cv2.IMREAD_COLOR)
        return cache[key]
    current_site = None
    for pid, t in enumerate(tr.sort_values(["site_id", "track_id"]).itertuples(), 1):
        site_no, site = t.site_id.split("_", 1)
        if t.site_id != current_site:
            cache.clear()
            current_site = t.site_id
        n_in_site[site] = n_in_site.get(site, 0) + 1
        ID = f"{pid:02d}"
        files = align[align.site_id == t.site_id].sort_index()
        files = files.assign(date=[f.rsplit("_", 1)[1].split(".")[0] for f in files.index]).sort_values("date")
        ref_file = t.ref_file
        ref = load(ref_file)
        H, W = ref.shape[:2]
        mpp = float(files.m_per_px.median()) if files.m_per_px.notna().any() else 0.143
        ref_c = crop(ref, t.ref_x, t.ref_y)
        ref_g = cv2.cvtColor(ref_c, cv2.COLOR_BGR2GRAY)
        ref_date = files.loc[ref_file, "date"]

        imgs, caps = [], []
        fn = f"{ID}_{ref_date}_current.jpg"
        save(out / "images" / fn, caption(mark(ref_c.copy()), f"{ref_date} - most recent image",
             "Red circle: the vessel"), a.quality, a.width)
        imgs.append(fn); caps.append(f"{ref_date} – most recent image (vessel circled)")
        seen = set(dets[dets.track_id == t.track_id].file)
        for f, r in files.iterrows():
            if f == ref_file:
                continue
            g = load(f, grey=True)
            state = "vessel detected" if f in seen else "vessel not detected"
            kinds = (("aligned", (r.dx, r.dy)),) if a.aligned_only else (("raw", (0, 0)), ("aligned", (r.dx, r.dy)))
            for kind, (dx, dy) in kinds:
                # pixel in this image that lands on the vessel's reference position
                c = crop(g, t.ref_x - dx, t.ref_y - dy)
                ov = mark(overlay(ref_g, c))
                text = f"{r.date} {kind}"
                sub = f"magenta = {r.date}, green = {ref_date} (reference); grey = unchanged   |   {state}"
                fn = f"{ID}_{r.date}_{kind}.jpg"
                save(out / "images" / fn, caption(ov, text, sub), a.quality, a.width)
                imgs.append(fn)
                caps.append(f"{r.date} {kind} – magenta {r.date}, green {ref_date}; {state}")

        lat, lon, acc = to_latlon(geo.get(site_no, geo.get(site_no.lstrip("0"), ("centre", np.nan, np.nan))),
                                  t.ref_x, t.ref_y, W, H, mpp)
        first = int(str(t.first_seen)[:4])
        last_img_year = int(str(ref_date)[:4])
        if t.still_present_latest:
            on_site = f"Since {first} ({last_img_year - first}+ years)"
        else:
            on_site = f"{first}–{str(t.last_seen)[:4]} (gone in later images)"
        pretty = str(site_names.get(site_no, site)).replace("-", "–")
        rows.append({
            "ID": ID, "Name": f"{pretty} – hull {n_in_site[site]}",
            "Status": ("Abandoned (verified)" if not a.auto else
                       "Likely abandoned (unverified)" if t.review_priority == 1 else
                       "Possibly abandoned (unverified)"),
            "Location": f"{pretty}, {DISTRICTS.get(site, '')}".rstrip(", "),
            "Latitude": round(lat, 6), "Longitude": round(lon, 6), "Coordinate accuracy": acc,
            "Vessel": f"[type?], ~{t.length_m:.0f} m",
            "Material": "",
            "On site": on_site,
            "Notes": "; ".join(x for x in [
                "" if pd.isna(t.review_condition) else f"Condition: {t.review_condition}",
                "" if pd.isna(t.review_notes) else str(t.review_notes)] if x),
            "Images": "; ".join(imgs), "Captions": " | ".join(caps),
            "Track ID": t.track_id,
        })
        print(f"{ID}  {t.track_id}  {len(imgs)} images  {lat:.5f},{lon:.5f}")

    df = pd.DataFrame(rows)
    with pd.ExcelWriter(out / "points.xlsx") as xw:
        df.to_excel(xw, sheet_name="Points", index=False)
        pd.DataFrame({"Column": ["Vessel", "Material", "Location", "Coordinates"],
                      "To do": ["Replace [type?] with the vessel type, e.g. 'Wooden fishing boat'",
                                "Fill in from the images, e.g. 'Timber hull, steel fittings, diesel engine'",
                                "District was pre-filled - please check",
                                "If 'approximate', add the Control points sheet and run the export again"]}
                     ).to_excel(xw, sheet_name="To do", index=False)
    print(f"\n{len(df)} vessels -> {out / 'points.xlsx'} and {out / 'images'}")


if __name__ == "__main__":
    main()

In [ ]:
import os
COORDS = '/content/drive/MyDrive/Abandoned_ships___image_coordinates.xlsx'   # your own file (used if step 8 was skipped)
COORDS_GEOREF = f'{OUT_DIR}/coordinates_georef.xlsx'      # from step 8 (falls back to your own file)
COORDS_USE = COORDS_GEOREF if os.path.exists(COORDS_GEOREF) else COORDS
print('using coordinates from', COORDS_USE)
WEB_DIR = f'{OUT_DIR}/website'
# NO_REVIEW = 0 : only boats you marked "yes" in tracks_reviewed.xlsx  -> Status "Abandoned (verified)"
# NO_REVIEW = 1 : skip review, export the strict group                 -> "Likely abandoned (unverified)"
# NO_REVIEW = 2 : skip review, strict group + "appearance changed" group -> also "Possibly abandoned (unverified)"
NO_REVIEW = 2
# GitHub Pages sites must stay under 1 GB. Several hundred boats with raw + aligned images is ~1.3 GB,
# so for a large export set ALIGNED_ONLY = True and/or lower QUALITY (65 still looks fine).
ALIGNED_ONLY = True    # True = skip the 'raw' images (half the files)
QUALITY = 60
WIDTH = 1000           # image width in px; 1500 = full detail, 1000 = about half the size
!rm -rf "$WEB_DIR"     # start clean, so old exports don't add to the size
!python export_website.py "$DATA_DIR" "$OUT_DIR" "$COORDS_USE" --out "$WEB_DIR" --auto $NO_REVIEW --quality $QUALITY --width $WIDTH {"--aligned-only" if ALIGNED_ONLY else ""}
!du -sh "$WEB_DIR/images"; ls "$WEB_DIR/images" | wc -l

## Settings you can change
Add them to the command in step 6, for example
`!python boat_pipeline.py "$DATA_DIR" --out "$OUT_DIR" --min-years 5 --match-m 10`

| Setting | Default | Meaning |
|---|---|---|
| `--min-years` | 5 | years in the same spot to count as abandoned (paper's rule) |
| `--match-m` | 8 | max movement in metres to count as "same position" |
| `--min-presence` | 0.6 | share of images (between first and last sighting) the boat must be detected in |
| `--conf` | 0.25 | detector confidence; lower = more boats found but more false detections |
| `--appearance-thr` | 0.55 | below this, a persistent spot is flagged as "maybe different boats" |

After changing `--match-m`, `--min-years`, `--min-presence` or `--appearance-thr` the run is fast
(detections are cached). After changing `--conf` or the model, delete the `cache` folder inside the results folder first.